# 🛡️ CyberAttack Dataset — Data Cleaning
**Goal:** Take the raw CSV, fix everything that'll cause problems in analysis or Power BI, and export a clean version.

---

## Step 0 — Paths & Imports

**Why:** Defining paths at the top means if you move files later, you only update one place instead of hunting through every cell. `os` gives us folder creation and file size checks. `numpy` handles some numeric operations pandas leans on.

In [ ]:
import pandas as pd
import numpy as np
import os

RAW_PATH      = r"C:\Users\Tarun\OneDrive\Desktop\College_PowerBi\Week_2_CyberAttack\Data\Cyberattacks_Detection.csv"
EXPORT_FOLDER = r"C:\Users\Tarun\OneDrive\Desktop\College_PowerBi\Week_2_CyberAttack\Outputs"

print("Libraries loaded. Paths set.")

## Step 1 — Load the raw data

**Why `low_memory=False`:** The dataset has columns like `Timestamp` and `Source Port` that have mixed types (numbers AND strings in the same column because of nulls). Without this flag, pandas guesses types chunk-by-chunk and throws a `DtypeWarning`. Setting it to False makes pandas read the whole file first, then decide — slower but accurate.

In [ ]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

## Step 2 — First look at the raw data

**Why:** Before touching anything, you want to know what you're dealing with. `head()` shows the actual values, `info()` shows dtypes and non-null counts, `isnull().sum()` gives a clean null count per column. You'll reference this throughout cleaning to verify each step worked.

In [ ]:
df.head(3)

In [ ]:
df.info()

In [ ]:
print("Null counts per column:")
df.isnull().sum()

### What we see in the raw data:
- **100,000 rows, 16 columns**
- `Attack ID`, `Source Port`, `Destination Port` loaded as **float64** — they should be integers. Pandas does this automatically when a numeric column has even one null, because Python integers can't hold NaN but floats can.
- `Timestamp` is 90% missing (90,174 nulls out of 100,000). The valid timestamps only exist in the last ~10k rows.
- Everything else has small, manageable null counts (~50–67 per column).
- `Port Type` is basically useless — we'll deal with that next.

## Step 3 — Drop the `Port Type` column

**Why:** `Port Type` has 99,922 rows with the value `'Other'` out of 100,000 rows. That's 99.9%. The remaining 78 rows have actual values like HTTP, SSH, FTP — but that's so few it tells you nothing useful. A column that's almost entirely one value adds zero signal to your analysis or dashboard. Keeping it would just be dead weight.

In [ ]:
print("Port Type distribution before dropping:")
print(df['Port Type'].value_counts(dropna=False))

df.drop(columns=['Port Type'], inplace=True)
print("\n✅ 'Port Type' column dropped.")
print(f"Shape is now: {df.shape[0]:,} rows × {df.shape[1]} columns")

## Step 4 — Rename columns properly

**Why:** The existing notebook used `.str.title()` for renaming, which is convenient but breaks abbreviations — `Source IP` becomes `Source Ip`, `ML Model` becomes `Ml Model`. That looks wrong and will confuse Power BI labels. We'll rename them manually so abbreviations stay correct and everything is clean and consistent.

In [ ]:
rename_map = {
    'Attack ID'           : 'Attack ID',
    'Source IP'           : 'Source IP',
    'Destination IP'      : 'Destination IP',
    'Source Country'      : 'Source Country',
    'Destination Country' : 'Destination Country',
    'Protocol'            : 'Protocol',
    'Source Port'         : 'Source Port',
    'Destination Port'    : 'Destination Port',
    'Attack Type'         : 'Attack Type',
    'Payload Size (bytes)': 'Payload Size (Bytes)',
    'Detection Label'     : 'Detection Label',
    'Confidence Score'    : 'Confidence Score',
    'ML Model'            : 'ML Model',
    'Affected System'     : 'Affected System',
    'Timestamp'           : 'Timestamp',
}
df.rename(columns=rename_map, inplace=True)

# Also strip any accidental whitespace from column names
df.columns = df.columns.str.strip()

print("Columns after renaming:")
print(list(df.columns))

## Step 5 — Fill missing values in categorical columns

**Why fill with `'Unknown'` instead of dropping:** These columns (Source Country, Protocol, Attack Type, etc.) each have only ~50–67 nulls out of 100,000. Dropping those rows would throw away valid data in all the other columns of that row. `'Unknown'` is a honest label — it tells Power BI that the value existed but wasn't recorded, which is different from the row not existing at all.

**Why `str.strip()`:** Catches any invisible leading/trailing spaces that would make `'India'` and `'India '` look like two different countries.

In [ ]:
cat_cols = [
    'Source IP', 'Destination IP',
    'Source Country', 'Destination Country',
    'Protocol', 'Attack Type',
    'Detection Label', 'ML Model', 'Affected System'
]

for col in cat_cols:
    null_count = df[col].isnull().sum()
    df[col] = df[col].fillna('Unknown').str.strip()
    print(f"{col:25s} → filled {null_count} nulls with 'Unknown'")

## Step 6 — Fill missing values in numeric columns

**Why fill with median (not mean):** `Source Port` ranges from 1 to 65,534 — a wide range. If there are any extreme values, the mean gets pulled toward them and becomes unrepresentative. Median always gives you the middle value regardless of extremes, which is a safer default for port numbers and scores.

**Why do this BEFORE type conversion:** In the original notebook, type conversion (to Int64) happened before null-filling. That works because Int64 is a nullable integer, but filling nulls on Int64 columns requires an extra cast. Filling first, converting after is the cleaner order.

In [ ]:
num_cols = {
    'Attack ID'           : 'median',
    'Source Port'         : 'median',
    'Destination Port'    : 'median',
    'Payload Size (Bytes)': 'median',
    'Confidence Score'    : 'median',
}

for col in num_cols:
    null_count = df[col].isnull().sum()
    fill_val   = df[col].median()
    df[col]    = df[col].fillna(fill_val)
    print(f"{col:25s} → filled {null_count} nulls with median ({fill_val:.1f})")

## Step 7 — Fix data types

**Why:** Now that nulls are filled, we can safely convert. `Attack ID`, `Source Port`, and `Destination Port` are whole numbers — storing them as float64 is wasteful and looks wrong (1.0 instead of 1). We convert to standard Python `int` now that there are no NaN values blocking us.

`Payload Size` and `Confidence Score` stay as float — they're legitimate decimal numbers.

In [ ]:
for col in ['Attack ID', 'Source Port', 'Destination Port']:
    df[col] = df[col].astype(int)

for col in ['Payload Size (Bytes)', 'Confidence Score']:
    df[col] = df[col].astype(float)

print("Dtypes after conversion:")
df[['Attack ID', 'Source Port', 'Destination Port', 'Payload Size (Bytes)', 'Confidence Score']].dtypes

## Step 8 — Handle the Timestamp column

**The situation:** 90,174 out of 100,000 rows have no timestamp. We can't fill these — we don't know when those attacks happened. Dropping those rows would lose 90% of the dataset.

**What we do instead:**
1. Parse the strings that DO exist into proper datetime objects (`pd.to_datetime`)
2. Leave the missing ones as `NaT` (Not a Time — datetime's equivalent of NaN)
3. Extract useful parts (hour, month, day of week) into separate columns — Power BI can filter and group by these even when the original timestamp is missing

**Why extract parts:** If you want to ask "do attacks spike at night?" you need the hour. Power BI can't extract that from a text field — it needs a dedicated column.

In [ ]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')

df['Attack Date']  = df['Timestamp'].dt.date
df['Attack Year']  = df['Timestamp'].dt.year
df['Attack Month'] = df['Timestamp'].dt.month_name()
df['Attack Hour']  = df['Timestamp'].dt.hour
df['Day Of Week']  = df['Timestamp'].dt.day_name()

print(f"Valid timestamps : {df['Timestamp'].notna().sum():,}")
print(f"Missing (NaT)    : {df['Timestamp'].isna().sum():,}")
print()
df[['Timestamp', 'Attack Date', 'Attack Month', 'Attack Hour', 'Day Of Week']].dropna().head(5)

## Step 9 — Add derived columns

These aren't fixes — they're new columns we create from existing data to make the dashboard more useful. Think of them as pre-computed answers to questions Power BI would struggle to compute on the fly.

**`Is Detected` (binary flag):**  
`Detection Label` is text ('Detected' / 'Not Detected'). Power BI can count text but can't average it. A 1/0 column lets you calculate detection rate as a simple average, use it in scatter plots, etc.

**`Payload Bucket`:**  
Payload Size is a continuous number (1 to 4,999 bytes). You can't put 4,999 unique values on a bar chart. Bucketing groups them into meaningful size ranges so charts are readable.

**`Confidence Tier`:**  
Same idea — Confidence Score is a decimal between 0 and 1. Low/Medium/High makes it usable as a slicer.

In [ ]:
# 1. Binary detection flag
df['Is Detected'] = df['Detection Label'].str.lower().map(
    {'detected': 1, 'not detected': 0, 'unknown': -1}
).fillna(-1).astype(int)

# 2. Payload size buckets
df['Payload Bucket'] = pd.cut(
    df['Payload Size (Bytes)'],
    bins  = [0, 500, 1000, 2000, 3000, 5001],
    labels= ['0–500 B', '500B–1KB', '1–2 KB', '2–3 KB', '3–5 KB']
)

# 3. Confidence score tier
df['Confidence Tier'] = pd.cut(
    df['Confidence Score'],
    bins  = [0, 0.4, 0.7, 1.01],
    labels= ['Low', 'Medium', 'High']
)

df[['Detection Label', 'Is Detected', 'Payload Size (Bytes)', 'Payload Bucket',
    'Confidence Score', 'Confidence Tier']].head(8)

## Step 10 — Drop duplicates

**Why:** Duplicate rows in a 100k dataset skew every count, average, and percentage in your dashboard. A bar chart that says 5,000 DNS Tunneling attacks might actually be showing the same attack recorded twice. We check and remove them before exporting.

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)  # resets row numbers cleanly after dropping

print(f"Duplicates removed : {before - len(df)}")
print(f"Final shape        : {df.shape[0]:,} rows × {df.shape[1]} columns")

## Step 11 — Final verification

**Why:** Always do a final check before exporting. You want to confirm: no unexpected nulls remain, dtypes look right, and the data looks sensible. If something's off, better to catch it here than after you've built half a Power BI dashboard on bad data.

In [ ]:
print("=== NULL CHECK ===")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "✅ No unexpected nulls (NaT in Timestamp is expected)")

print("\n=== DTYPE CHECK ===")
print(df.dtypes)

print("\n=== SAMPLE ===")
df.head(3)

## Step 12 — Export cleaned CSV

**`index=False`:** By default, pandas would add a column called `index` (0, 1, 2...) to the CSV. We don't want that — the row numbers aren't data, they're just pandas internals.

`os.makedirs(..., exist_ok=True)` creates the Outputs folder if it doesn't exist yet, without crashing if it already does.

In [ ]:
os.makedirs(EXPORT_FOLDER, exist_ok=True)

out_path = os.path.join(EXPORT_FOLDER, 'Cyberattacks_Cleaned.csv')
df.to_csv(out_path, index=False)

print(f"✅ Saved to : {out_path}")
print(f"   Size    : {os.path.getsize(out_path) / 1_000_000:.1f} MB")
print(f"   Rows    : {len(df):,}")
print(f"   Columns : {df.shape[1]}")